In [1]:
import pandas as pd

# Carregar o arquivo CSV
csv_file = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/1missing.csv'
df_csv = pd.read_csv(csv_file)

# Carregar o arquivo Excel (dicionário)
excel_file = '/Users/marcelosilva/Desktop/early-obesity-prediction/4-Dicionario-ENANI-2019 (1).xlsx'
df_dict = pd.read_excel(excel_file)

# Obter todas as colunas do CSV
csv_columns = df_csv.columns.tolist()

# Criar lista para armazenar os matches
matches = []

# Cruzar os nomes das colunas com as variáveis do dicionário
for column in csv_columns:
    # Procurar por matches (case insensitive)
    match = df_dict[df_dict['variavel'].str.lower() == column.lower()]
    
    if not match.empty:
        # Se encontrou match, adicionar à lista
        description = match['descricao'].iloc[0] if 'descricao' in df_dict.columns else 'Descrição não encontrada'
        matches.append(f"{column}: {description}")

# Salvar os resultados no arquivo 2dict.txt
output_file = '2dict.txt'
with open(output_file, 'w', encoding='utf-8') as f:
    for match in matches:
        f.write(match + '\n')

print(f"Arquivo {output_file} criado com {len(matches)} correlações encontradas.")
print(f"Total de colunas no CSV: {len(csv_columns)}")

Arquivo 2dict.txt criado com 419 correlações encontradas.
Total de colunas no CSV: 419


/opt/anaconda3/envs/pycaret_env/lib/python3.8/site-packages/openpyxl/worksheet/_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


In [6]:
import pandas as pd
import os

# Lista de variáveis selecionadas para análise dos primeiros 24 meses de vida
selected_variables = [
    # Identificação e características demográficas
    'id_anon',                    # Código de identificação da criança
    'a00_regiao',                 # Macrorregião 
    'b02_sexo',                   # Sexo da criança 
    'b04_idade',                  # Idade em anos completos da criança
    'bb04_idade_da_mae',          # Idade em anos completos da mãe
    'd01_cor',                    # Cor ou raça da criança
    
    # Dados perinatais e nascimento
    'h01_semanas_gravidez',       # Com quantas semanas de gravidez a criança nasceu
    'h02_peso',                   # Peso ao nascer (em gramas) da criança
    'h03_altura',                 # Com quantos centímetros a criança nasceu
    'h04_parto',                  # Tipo de parto da criança
    'h05_chupeta_usou',           # Desde o nascimento da criança até hoje, ele(ela)
    
    # Condições congênitas e síndromes
    'h10b_sindrome_de_down',      # Algum médico já falou que a criança tem síndrome de Down, fibrose cística, fenilcetonúria ou autismo
    'h10b1_sindrome_nao',         # Não tem síndrome
    'h10b2_sindrome_down',        # Síndrome de Down
    'h10b3_sindrome_fibrose',     # Fibrose cística
    'h10b4_sindrome_fenilcetonuria', # Fenilcetonúria
    'h10b5_sindrome_autismo',     # Autismo
    'h10b9_sindrome_nao_sabe',    # Não sabe
    
    # Características maternas - educação
    'j08_ler',                    # A mãe/responsável sabe ler e escrever
    'j09_frequenta',              # A mãe/responsável frequenta ou já frequentou a escola
    'j10_serie',                  # Última série/ano que a mãe/responsável completou
    
    # Pré-natal e gestação
    'k03_prenatal',               # Fez o pré-natal da criança
    'k04_prenatal_semanas',       # Com quantas semanas de gravidez você iniciou o pré-natal da criança
    'k05_prenatal_consultas',     # Quantas consultas de pré-natal teve durante a gravidez da criança
    'k06_peso_engravidar',        # Qual era o seu peso antes de engravidar da criança
    'k07_peso_final',             # Qual era o seu peso antes do parto (ao final da gestação) da criança
    'k08_quilos',                 # Quantos quilos você ganhou na gestação da criança
    
    # Aleitamento materno e práticas alimentares iniciais
    'k11_amamentou',              # Amamentou alguma vez a criança
    'k12_tempo',                  # Quanto tempo depois do nascimento a criança foi colocada no peito pela primeira vez para mamar
    'k13_tempo_medida',           # Horas ou dias
    'k15_recebeu',                # Durante sua permanência na maternidade a criança recebeu algum outro leite que não fosse o leite de peito
    'k16_liquido',                # Nos primeiros dias após o parto, e antes que o seu leite estivesse descendo, foi dado algum outro líquido para a criança beber que não fosse leite materno
    'k18_somente',                # Durante quanto tempo deu somente leite do peito para a criança, sem água, chá, suco ou outros alimentos
    'k19_somente_medida',         # Dias ou meses
    'k20_doou',                   # Desde que amamenta a criança doou o seu leite a algum banco de leite humano ou posto de coleta de leite humano
    'k21_recebeu',                # Desde que amamenta a criança alguma vez recebeu leite de um banco de leite humano
    'k22_amamentou',              # Desde que amamenta a criança você amamentou o filho de outra mulher
    'k23_deixou',                 # Desde que amamenta a criança deixou seu filho ser amamentado por outra mulher
    'k24_utilizou',               # Desde que amamenta a criança utilizou
    'k241_utilizou_concha',       # Utilizou concha
    'k242_utilizou_protetor',     # Utilizou protetor
    'k243_utilizou_bico',         # Utilizou bico
    'k244_utilizou_bomba',        # Utilizou bomba
    'k245_utilizou_mamadeira',    # Utilizou mamadeira
    'k246_utilizou_sondinha',     # Utilizou sondinha
    'k247_utilizou_copo',         # Utilizou copo
    'k248_utilizou_nao',          # Não utilizou nenhum
    'k249_utilizou_nao_sabe',     # Não sabe
    'k25_mamadeira',              # A criança usa ou já usou mamadeira
    'k28_aleitamento',            # Você acessa ou acessou a internet para buscar informações sobre aleitamento materno
    
    # Variáveis socioeconômicas e de equidade
    'q01_recebe_beneficio',       # Alguém do domicílio recebe benefício(s) do governo
    
    # Antropometria materna atual (para feature engineering)
    't02_peso_medida1',           # Medida 1 de peso da mãe (kg)
    't03_peso_medida2',           # Medida 2 de peso da mãe (kg)
    
    # Indicadores socioeconômicos
    'vd_ien_escore',              # Escore do Indicador Econômico Nacional (IEN)
    
    # Variáveis target/outcome
    'vd_zimc',                    # Escore z do índice IMC/idade da criança
    'vd_prematura_igb'            # Crianças prematuras e com idade gestacional entre 189 e 454 dias
]

def filter_dataset():
    """
    Filtra o dataset original mantendo apenas as variáveis selecionadas
    e gera relatório das alterações
    """
    
    # Definir caminhos
    input_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/1missing.csv'
    output_csv_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/2features24.csv'
    output_txt_path = '/Users/marcelosilva/Desktop/early-obesity-prediction/B-Featuring Removing/2features24.txt'
    
    try:
        # Carregar o dataset original
        print("Carregando dataset original...")
        df_original = pd.read_csv(input_path)
        
        # Verificar quais variáveis selecionadas existem no dataset
        available_vars = [var for var in selected_variables if var in df_original.columns]
        missing_vars = [var for var in selected_variables if var not in df_original.columns]
        
        # Filtrar dataset com variáveis disponíveis
        df_filtered = df_original[available_vars].copy()
        
        # Salvar novo CSV
        print("Salvando dataset filtrado...")
        df_filtered.to_csv(output_csv_path, index=False)
        
        # Calcular estatísticas
        total_original = len(df_original.columns)
        total_selecionadas = len(available_vars)
        total_excluidas = total_original - total_selecionadas
        
        # Identificar variáveis excluídas
        excluded_vars = [col for col in df_original.columns if col not in available_vars]
        
        # Criar relatório em arquivo TXT
        print("Gerando relatório...")
        with open(output_txt_path, 'w', encoding='utf-8') as f:
            f.write("RELATÓRIO DE SELEÇÃO DE VARIÁVEIS - PRIMEIROS 24 MESES DE VIDA\n")
            f.write("=" * 70 + "\n\n")
            
            f.write("RESUMO ESTATÍSTICO:\n")
            f.write("-" * 20 + "\n")
            f.write(f"Total de colunas no dataset original (1missing.csv): {total_original}\n")
            f.write(f"Total de variáveis selecionadas: {len(selected_variables)}\n")
            f.write(f"Total de variáveis disponíveis e mantidas: {total_selecionadas}\n")
            f.write(f"Total de variáveis excluídas: {total_excluidas}\n")
            f.write(f"Dimensões do dataset original: {df_original.shape}\n")
            f.write(f"Dimensões do dataset filtrado: {df_filtered.shape}\n\n")
            
            if missing_vars:
                f.write("VARIÁVEIS SELECIONADAS MAS NÃO ENCONTRADAS NO DATASET:\n")
                f.write("-" * 50 + "\n")
                for i, var in enumerate(missing_vars, 1):
                    f.write(f"{i:2d}. {var}\n")
                f.write("\n")
            
            f.write("VARIÁVEIS MANTIDAS NO DATASET FILTRADO:\n")
            f.write("-" * 40 + "\n")
            for i, var in enumerate(available_vars, 1):
                f.write(f"{i:2d}. {var}\n")
            
            f.write(f"\n\nVARIÁVEIS EXCLUÍDAS DO DATASET ORIGINAL ({total_excluidas}):\n")
            f.write("-" * 50 + "\n")
            for i, var in enumerate(excluded_vars, 1):
                f.write(f"{i:3d}. {var}\n")
        
        # Resumo final
        print("\n" + "="*50)
        print("PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
        print("="*50)
        print(f"Dataset original: {df_original.shape[0]} linhas × {df_original.shape[1]} colunas")
        print(f"Dataset filtrado: {df_filtered.shape[0]} linhas × {df_filtered.shape[1]} colunas")
        print(f"Variáveis removidas: {total_excluidas}")
        print(f"\nArquivos gerados:")
        print(f"• CSV filtrado: 2features24.csv")
        print(f"• Relatório: 2features24.txt")
        
        if missing_vars:
            print(f"\n⚠️  ATENÇÃO: {len(missing_vars)} variável(es) selecionada(s) não foi(ram) encontrada(s) no dataset original.")
            print("Verifique o relatório para detalhes.")
        
        return df_filtered
        
    except FileNotFoundError:
        print(f"❌ Erro: Arquivo não encontrado em {input_path}")
        print("Verifique se o caminho está correto.")
        return None
        
    except Exception as e:
        print(f"❌ Erro durante o processamento: {str(e)}")
        return None

# Executar o processamento
if __name__ == "__main__":
    df_resultado = filter_dataset()

Carregando dataset original...
Salvando dataset filtrado...
Gerando relatório...

PROCESSAMENTO CONCLUÍDO COM SUCESSO!
Dataset original: 8236 linhas × 419 colunas
Dataset filtrado: 8236 linhas × 56 colunas
Variáveis removidas: 363

Arquivos gerados:
• CSV filtrado: 2features24.csv
• Relatório: 2features24.txt


## 🔍 Variable Selection for First 24 Months of Life Analysis

This cell implements a function to **filter the original dataset** keeping only the most relevant variables for analyzing child development in the first 24 months of life.

### 📋 Selected Variables

The set of 56 variables was carefully chosen covering:

- **Identification and Demographics**: Child ID, region, sex, age, maternal characteristics
- **Perinatal Data**: gestational weeks, birth weight/height, delivery type
- **Congenital Conditions**: syndromes and special conditions
- **Maternal Characteristics**: education, literacy
- **Prenatal and Pregnancy**: consultations, maternal weight, weight gain
- **Breastfeeding**: breastfeeding and early feeding practices
- **Socioeconomic Indicators**: benefits and economic status
- **Anthropometry and Outcomes**: maternal BMI, child BMI z-scores, prematurity indicators

### 🎯 Main Features

- **Comprehensive Coverage**: Variables spanning from pregnancy to 24 months
- **Evidence-Based Selection**: Focus on factors known to influence early childhood development
- **Quality Control**: Automated filtering with detailed reporting
- **Output Generation**: Creates both filtered dataset (2features24.csv) and detailed report (2features24.txt)